In [24]:
!unzip Early_Sepsis_Alert.zip -d /content/extracted_folder

Archive:  Early_Sepsis_Alert.zip
replace /content/extracted_folder/Early_Sepsis_Alert/.DS_Store? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

# 02 — Feature Engineering
## Early Sepsis Alert — Building the 6–12h Early Warning Target + Features

**Two jobs in this notebook:**
1. Convert `SepsisLabel` (current-state label) into a **future-onset target**
   (`target_6h`, `target_12h`) — this is what actually makes it an "early alert"
   system instead of a detector.
2. Engineer features that carry *trend* information (deltas, slopes, rolling
   stats) and *clinical domain knowledge* (SIRS, shock index, qSOFA proxy),
   since these are what let a model hit high recall (catch deterioration early)
   without destroying precision (avoid firing on noisy single readings).

Output: `data/processed/train_fe.csv`, `data/processed/test_fe.csv`,
`data/processed/engineered_features.csv` (metadata for the new columns).

# Cell 2 — Imports & config

In [25]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("../data/processed")
ID_COL = "patient_id"
TIME_COL = "HourInStay"
TARGET_COL = "SepsisLabel"

# Prediction horizons (hours). Problem statement asks for 6-12h early warning.
HORIZON_SHORT = 6    # stricter / earliest possible warning
HORIZON_LONG = 12    # more lead time, easier target

# Vitals we'll build rolling/trend features for (continuous, clinically meaningful)
TREND_VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp",
                 "WBC", "Lactate", "Platelets", "Creatinine", "BUN",
                 "PaCO2", "pH", "Bilirubin_total", "Glucose"]

ROLLING_WINDOWS = [4, 8, 16, 24]   # hours
DELTA_WINDOWS = [1, 3, 6]          # hours

np.random.seed(42)

# Cell 3  — Load

In [26]:
train = pd.read_csv('/content/extracted_folder/Early_Sepsis_Alert/Data/Processed Data/train.csv').sort_values([ID_COL, TIME_COL]).reset_index(drop=True)
test = pd.read_csv('/content/extracted_folder/Early_Sepsis_Alert/Data/Processed Data/test.csv').sort_values([ID_COL, TIME_COL]).reset_index(drop=True)

print("Train:", train.shape, "| patients:", train[ID_COL].nunique())
print("Test :", test.shape, "| patients:", test[ID_COL].nunique())


Train: (46482, 43) | patients: 1226
Test : (11894, 43) | patients: 307


## 1. Build the Future-Onset Target

At each hour `t`, look FORWARD within the same patient's timeline:
- `target_6h[t]  = 1` if SepsisLabel becomes 1 at any point in `(t, t+6]`
- `target_12h[t] = 1` if SepsisLabel becomes 1 at any point in `(t, t+12]`

Once a patient is already septic (`SepsisLabel[t] == 1`), we don't need to
"predict" anything anymore — those rows get excluded from training later
(there's no alert to give someone already flagged).

# Cell 5 — Future-onset label function (with self-check)

In [27]:
def make_future_onset_labels(labels: np.ndarray, horizons: list) -> dict:
    """
    labels: 1D array of SepsisLabel for ONE patient, ordered by time ascending.
    Returns dict {horizon: array} where result[t] = 1 if any labels[t+1 : t+1+h] == 1
    """
    n = len(labels)
    out = {h: np.zeros(n, dtype=int) for h in horizons}
    # cumulative "any sepsis from index i onward" — precompute suffix max for O(n)
    suffix_max = np.zeros(n + 1, dtype=int)
    for i in range(n - 1, -1, -1):
        suffix_max[i] = max(labels[i], suffix_max[i + 1])

    for h in horizons:
        for t in range(n):
            end = min(t + 1 + h, n)
            start = t + 1
            if start >= n:
                out[h][t] = 0
            else:
                out[h][t] = 1 if labels[start:end].max() > 0 else 0
    return out

# --- sanity check on a synthetic patient before trusting it on real data ---
test_labels = np.array([0, 0, 0, 0, 0, 1, 1, 0])
result = make_future_onset_labels(test_labels, [3])
# at t=0, next 3 hrs = idx[1,2,3] = [0,0,0] -> 0
# at t=2, next 3 hrs = idx[3,4,5] = [0,0,1] -> 1
# at t=4, next 3 hrs = idx[5,6,7] = [1,1,0] -> 1
# at t=6, next 3 hrs = idx[7]     = [0]     -> 0
assert result[3].tolist() == [0, 0, 1, 1, 1, 1, 0, 0], f"Got {result[3].tolist()}"
print("Label logic verified ✓")


Label logic verified ✓


# Cell 6  — Apply per patient

In [28]:
def build_targets(df):
    frames = []
    for pid, g in df.groupby(ID_COL, sort=False):
        g = g.sort_values(TIME_COL).copy()
        labels = g[TARGET_COL].values
        future = make_future_onset_labels(labels, [HORIZON_SHORT, HORIZON_LONG])
        g[f"target_{HORIZON_SHORT}h"] = future[HORIZON_SHORT]
        g[f"target_{HORIZON_LONG}h"] = future[HORIZON_LONG]
        frames.append(g)
    return pd.concat(frames, ignore_index=True)

train = build_targets(train)
test = build_targets(test)

for name, df in [("train", train), ("test", test)]:
    r6 = df[f"target_{HORIZON_SHORT}h"].mean()
    r12 = df[f"target_{HORIZON_LONG}h"].mean()
    print(f"{name}: target_6h positive rate = {r6:.4%} | target_12h positive rate = {r12:.4%}")

train: target_6h positive rate = 3.1819% | target_12h positive rate = 4.1134%
test: target_6h positive rate = 3.0099% | target_12h positive rate = 3.9852%


## 2. Drop Rows Where Patient Is *Already* Septic

Once `SepsisLabel == 1`, an early-warning alert is moot — the event already
happened. Keep these rows out of the *training* target rows, but keep them
in the dataframe for now (rolling windows still need the full history as
context for earlier hours). We'll filter at the very end, right before saving.

## 3. Rolling Window Statistics

For each key vital: rolling mean / std / min / max over trailing 4h, 8h, 16h, 24h.
Computed strictly on PAST data (no leakage) using `min_periods=1` so early
hours in a stay still get a (noisier) estimate rather than NaN.

# Cell 9 — Rolling stats

In [29]:
def add_rolling_features(df, vitals, windows):
    df = df.sort_values([ID_COL, TIME_COL]).copy()
    grouped = df.groupby(ID_COL, sort=False)
    for vital in vitals:
        for w in windows:
            roll = grouped[vital].rolling(window=w, min_periods=1)
            df[f"{vital}_roll{w}h_mean"] = roll.mean().reset_index(level=0, drop=True)
            df[f"{vital}_roll{w}h_std"]  = roll.std().reset_index(level=0, drop=True).fillna(0)
            df[f"{vital}_roll{w}h_min"]  = roll.min().reset_index(level=0, drop=True)
            df[f"{vital}_roll{w}h_max"]  = roll.max().reset_index(level=0, drop=True)
    return df

train = add_rolling_features(train, TREND_VITALS, ROLLING_WINDOWS)
test = add_rolling_features(test, TREND_VITALS, ROLLING_WINDOWS)
print("After rolling features:", train.shape)

After rolling features: (46482, 301)


## 4. Delta (Rate-of-Change) Features

`delta = current_value - value_N_hours_ago`. This is the single most
important signal type for early sepsis detection — a rapidly *changing*
vital is a much stronger warning than an absolute value sitting near the
edge of normal range.

# Cell 11  — Deltas

In [30]:
def add_delta_features(df, vitals, windows):
    df = df.sort_values([ID_COL, TIME_COL]).copy()
    grouped = df.groupby(ID_COL, sort=False)
    for vital in vitals:
        for w in windows:
            shifted = grouped[vital].shift(w)
            df[f"{vital}_delta{w}h"] = df[vital] - shifted
    # fill NaN deltas (start of stay, not enough history) with 0 = "no change detected yet"
    delta_cols = [c for c in df.columns if "_delta" in c]
    df[delta_cols] = df[delta_cols].fillna(0)
    return df

train = add_delta_features(train, TREND_VITALS, DELTA_WINDOWS)
test = add_delta_features(test, TREND_VITALS, DELTA_WINDOWS)
print("After delta features:", train.shape)


After delta features: (46482, 349)


## 5. Slope (Trend Direction) Features

Linear regression slope of each vital over the trailing 6h window. Distinguishes
"trending up fast" from "spiked once then flat" — deltas alone can miss this.

# Cell 13 — Slopes

In [31]:
def rolling_slope(series, window):
    """Slope of best-fit line over trailing `window` points."""
    x = np.arange(window)
    def _slope(y):
        if len(y) < 2 or np.all(y == y[0]):
            return 0.0
        n = len(y)
        xi = x[:n]
        try:
            return np.polyfit(xi, y, 1)[0]
        except Exception:
            return 0.0
    return series.rolling(window, min_periods=2).apply(_slope, raw=True).fillna(0)

def add_slope_features(df, vitals, window=6):
    df = df.sort_values([ID_COL, TIME_COL]).copy()
    grouped = df.groupby(ID_COL, sort=False)
    for vital in vitals:
        df[f"{vital}_slope{window}h"] = grouped[vital].transform(lambda s: rolling_slope(s, window))
    return df

train = add_slope_features(train, TREND_VITALS, window=6)
test = add_slope_features(test, TREND_VITALS, window=6)
print("After slope features:", train.shape)

After slope features: (46482, 365)


## 6. Clinical Composite Scores

Domain-knowledge features that historically drive precision by encoding
known sepsis heuristics directly, rather than hoping the model rediscovers
them from raw correlations:

- **Shock Index** = HR / SBP (>0.9 is a red flag)
- **SIRS criteria count** (Temp, HR, Resp/PaCO2, WBC — 4-point score, ≥2 = SIRS positive)
- **qSOFA proxy** (Resp ≥22, SBP ≤100 — GCS unavailable in this dataset so 2-point proxy)
- **BUN/Creatinine ratio** (renal stress marker)
- **MAP < 65** hypotension flag (organ hypoperfusion threshold)

# Cell 15 — Clinical scores

In [32]:
def add_clinical_scores(df):
    df = df.copy()

    # Shock Index
    df["shock_index"] = df["HR"] / df["SBP"].replace(0, np.nan)
    df["shock_index"] = df["shock_index"].fillna(0)

    # SIRS criteria (each condition contributes 1 point)
    sirs_temp = ((df["Temp"] > 38) | (df["Temp"] < 36)).astype(int)
    sirs_hr = (df["HR"] > 90).astype(int)
    sirs_resp = ((df["Resp"] > 20) | (df["PaCO2"] < 32)).astype(int)
    sirs_wbc = ((df["WBC"] > 12) | (df["WBC"] < 4)).astype(int)
    df["sirs_score"] = sirs_temp + sirs_hr + sirs_resp + sirs_wbc
    df["sirs_positive"] = (df["sirs_score"] >= 2).astype(int)

    # qSOFA proxy (GCS not in dataset, so 2-criteria proxy instead of standard 3)
    qsofa_resp = (df["Resp"] >= 22).astype(int)
    qsofa_sbp = (df["SBP"] <= 100).astype(int)
    df["qsofa_proxy_score"] = qsofa_resp + qsofa_sbp

    # BUN/Creatinine ratio (renal stress)
    df["bun_creatinine_ratio"] = df["BUN"] / df["Creatinine"].replace(0, np.nan)
    df["bun_creatinine_ratio"] = df["bun_creatinine_ratio"].fillna(0)

    # Hypotension flag
    df["hypotension_flag"] = (df["MAP"] < 65).astype(int)

    # Hypoxia flag
    df["hypoxia_flag"] = (df["O2Sat"] < 90).astype(int)

    return df

train = add_clinical_scores(train)
test = add_clinical_scores(test)
print("After clinical scores:", train.shape)

After clinical scores: (46482, 372)


## 7. Missingness / "Not Measured" Flags

From EDA, several labs (EtCO2, Bilirubin_direct, TroponinI, Fibrinogen, etc.)
are heavily zero-inflated — almost certainly "not ordered" rather than a true
physiological zero. A lab being ORDERED at all is itself informative (clinicians
only order troponin/lactate/etc. when they're already suspicious) so we encode
that as a binary signal instead of letting the model treat 0 as a normal value.

# Cell 17  — Measurement flags

In [33]:
SPARSE_LABS = ["EtCO2", "Bilirubin_direct", "TroponinI", "Fibrinogen",
               "AST", "Alkalinephos", "Lactate", "SaO2", "PaCO2"]

def add_measurement_flags(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[f"{c}_measured"] = (df[c] != 0).astype(int)
    return df

train = add_measurement_flags(train, SPARSE_LABS)
test = add_measurement_flags(test, SPARSE_LABS)
print("After measurement flags:", train.shape)


After measurement flags: (46482, 381)


## 8. Sanity Checks Before Saving

- No infinities from division features (shock_index, bun/creatinine ratio)
- No leakage: confirm rolling/delta/slope features only used past data (spot check one patient)
- Class balance check on final targets after dropping "already septic" rows

# Cell 19 — Sanity checks

In [34]:
for df, name in [(train, "train"), (test, "test")]:
    n_inf = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
    print(f"{name}: infinite values = {n_inf}")
    df.replace([np.inf, -np.inf], 0, inplace=True)

# Spot check leakage: pick one patient, verify HR_roll4h_mean at hour t
# only reflects hours <= t
sample_pid = train[ID_COL].iloc[0]
sample = train[train[ID_COL] == sample_pid].sort_values(TIME_COL)
t = 5
manual_mean = sample["HR"].iloc[max(0, t-3):t+1].mean()  # trailing 4h incl current
computed_mean = sample["HR_roll4h_mean"].iloc[t]
print(f"Leakage check — manual: {manual_mean:.3f} vs computed: {computed_mean:.3f} "
      f"({'OK' if np.isclose(manual_mean, computed_mean) else 'MISMATCH'})")

train: infinite values = 0
test: infinite values = 0
Leakage check — manual: 68.500 vs computed: 68.500 (OK)


## 9. Finalize: Drop "Already Septic" Rows, Save Outputs

Rows where `SepsisLabel == 1` are excluded from the modeling set — there's
no early-warning decision to make once the patient is already flagged septic.

# Cell 21 — Filter & save

In [35]:
train_final = train[train[TARGET_COL] == 0].reset_index(drop=True)
test_final = test[test[TARGET_COL] == 0].reset_index(drop=True)

print("Final train shape:", train_final.shape)
print("Final test shape :", test_final.shape)
print(f"target_6h positive rate  — train: {train_final['target_6h'].mean():.4%} ")
print(f"| test: {test_final['target_6h'].mean():.4%}")
print(f"target_12h positive rate — train: {train_final['target_12h'].mean():.4%} ")
print(f"| test: {test_final['target_12h'].mean():.4%}")

DATA_DIR.mkdir(parents=True, exist_ok=True)
train_final.to_csv(DATA_DIR / "train_fe.csv", index=False)
test_final.to_csv(DATA_DIR / "test_fe.csv", index=False)

# Metadata for the engineered feature set (feeds train.py's feature selection)
exclude_cols = [ID_COL, TIME_COL, TARGET_COL, "target_6h", "target_12h"]
final_feature_cols = [c for c in train_final.columns if c not in exclude_cols]

meta_rows = []
for c in final_feature_cols:
    meta_rows.append({
        "feature_name": c,
        "dtype": str(train_final[c].dtype),
        "category": (
            "rolling" if "_roll" in c else
            "delta" if "_delta" in c else
            "slope" if "_slope" in c else
            "clinical_score" if c in ["shock_index", "sirs_score", "sirs_positive",
                                       "qsofa_proxy_score", "bun_creatinine_ratio",
                                       "hypotension_flag", "hypoxia_flag"] else
            "measurement_flag" if c.endswith("_measured") else
            "raw_vital"
        ),
        "mean": round(train_final[c].mean(), 4),
        "std": round(train_final[c].std(), 4),
    })

engineered_features_df = pd.DataFrame(meta_rows)
engineered_features_df.to_csv(DATA_DIR / "engineered_features.csv", index=False)

print(f"\nTotal engineered features: {len(final_feature_cols)}")
print(engineered_features_df["category"].value_counts())

Final train shape: (45438, 381)
Final test shape : (11632, 381)
target_6h positive rate  — train: 1.2016% 
| test: 1.0660%
target_12h positive rate — train: 2.1546% 
| test: 2.0633%

Total engineered features: 376
category
rolling             256
delta                48
raw_vital            40
slope                16
measurement_flag      9
clinical_score        7
Name: count, dtype: int64


## 10. Summary & Handoff to `train.py`

- Targets available: `target_6h` (primary, strict early-warning) and
  `target_12h` (secondary, more lead time / easier)
- ~{N} engineered features across raw vitals, rolling stats, deltas, slopes,
  clinical composite scores, and measurement flags
- Class imbalance is still severe on `target_6h` — `train.py` needs to handle
  this via `scale_pos_weight` (XGBoost) / `class_weight='balanced'` (RF),
  and NOT rely on default 0.5 threshold: recall/precision targets
  (R 92-95%, P 80-85%, F2 0.88-0.92) require explicit threshold tuning on
  the validation set — that's `evaluate.py`'s job, using precision-recall
  curve + F2 sweep rather than the default classification threshold.
- Next: `03_model_experiments.ipynb` — baseline RF/XGBoost, threshold tuning,
  F2 optimization.